**Model Architecture**

In [7]:
import torch
import torch.nn as nn
import torch.nn.functional as F


# =========================================================
# BAYESIAN LINEAR LAYER
# =========================================================
class BayesianLinear(nn.Module):
    def __init__(self, in_features: int, out_features: int, prior_std: float = 1.0):
        super().__init__()

        self.in_features = in_features
        self.out_features = out_features
        self.prior_std = prior_std

        # Mean parameters
        self.weight_mean = nn.Parameter(torch.randn(out_features, in_features) * 0.1)
        self.bias_mean = nn.Parameter(torch.randn(out_features) * 0.1)

        # Log variance (log σ²)
        self.weight_logvar = nn.Parameter(torch.randn(out_features, in_features) * 0.1 - 5)
        self.bias_logvar = nn.Parameter(torch.randn(out_features) * 0.1 - 5)

    def forward(self, x: torch.Tensor, sample: bool = True) -> torch.Tensor:
        if sample:
            weight_std = torch.exp(0.5 * self.weight_logvar)
            bias_std = torch.exp(0.5 * self.bias_logvar)

            weight = self.weight_mean + weight_std * torch.randn_like(weight_std)
            bias = self.bias_mean + bias_std * torch.randn_like(bias_std)
        else:
            weight = self.weight_mean
            bias = self.bias_mean

        return F.linear(x, weight, bias)

    def get_kl_divergence(self) -> torch.Tensor:
        """
        KL divergence between:
        q(w|θ) = N(μ, σ²)
        p(w)   = N(0, prior_std²)
        """
        weight_var = torch.exp(self.weight_logvar)
        bias_var = torch.exp(self.bias_logvar)

        prior_var = torch.tensor(self.prior_std ** 2, device=weight_var.device)

        kl_weight = 0.5 * torch.sum(
            (weight_var + self.weight_mean**2) / prior_var
            - 1.0
            + torch.log(prior_var)
            - self.weight_logvar
        )

        kl_bias = 0.5 * torch.sum(
            (bias_var + self.bias_mean**2) / prior_var
            - 1.0
            + torch.log(prior_var)
            - self.bias_logvar
        )

        return kl_weight + kl_bias


# =========================================================
# BAYESIAN HEAD
# =========================================================
class BayesianDetectionHead(nn.Module):
    def __init__(self, input_channels: int = 512):
        super().__init__()

        self.fc1 = BayesianLinear(input_channels, 256)
        self.fc2 = BayesianLinear(256, 128)

        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.3)

    def forward(self, x: torch.Tensor, sample: bool = True) -> torch.Tensor:
        x = self.fc1(x, sample)
        x = self.relu(x)
        x = self.dropout(x)

        x = self.fc2(x, sample)
        x = self.relu(x)
        x = self.dropout(x)

        return x

    def get_kl_divergence(self) -> torch.Tensor:
        return self.fc1.get_kl_divergence() + self.fc2.get_kl_divergence()


# =========================================================
# B-YOLO MODEL
# =========================================================
class B_YOLO(nn.Module):
    def __init__(self, backbone: nn.Module, num_classes: int = 80):
        super().__init__()

        self.backbone = backbone
        self.num_classes = num_classes

        # Freeze backbone properly ✔️
        for p in self.backbone.parameters():
            p.requires_grad = False

        self.bayesian_head = BayesianDetectionHead(512)

        # Task heads
        self.detection_head = nn.Linear(128, 4 + num_classes)
        self.segmentation_head = nn.Linear(128, 256)
        self.pose_head = nn.Linear(128, 17 * 2)
        self.oriented_head = nn.Linear(128, 5 + num_classes)
        self.classification_head = nn.Linear(128, num_classes)

        self.current_task = None

    def set_task(self, task: str):
        valid_tasks = ['detection', 'segmentation', 'pose', 'oriented', 'classification']
        if task not in valid_tasks:
            raise ValueError(f"Task must be one of {valid_tasks}")
        self.current_task = task

    def forward(
        self,
        x: torch.Tensor,
        sample: bool = True,
        num_mc_samples: int = 1,
        return_uncertainty: bool = True
    ) -> dict:
        """
        Forward pass with optional uncertainty estimation.

        Args:
            x: Input tensor
            sample: Whether to sample from Bayesian layers
            num_mc_samples: Number of MC samples for uncertainty (if > 1)
            return_uncertainty: Whether to compute and return uncertainty

        Returns:
            dict with 'predictions' and 'uncertainty' keys
        """
        # Backbone (with no grad)
        with torch.no_grad():
            features = self.backbone(x)

        # Flatten
        if features.dim() == 4:
            features = F.adaptive_avg_pool2d(features, 1)
            features = features.view(features.size(0), -1)

        # Compute predictions with uncertainty via MC sampling
        if return_uncertainty and num_mc_samples > 1:
            predictions_list = []

            for _ in range(num_mc_samples):
                # Bayesian head with stochastic sampling
                z = self.bayesian_head(features, sample=sample)

                # Task-specific output
                if self.current_task == 'detection':
                    pred = self.detection_head(z)
                elif self.current_task == 'segmentation':
                    pred = self.segmentation_head(z)
                elif self.current_task == 'pose':
                    pred = self.pose_head(z)
                elif self.current_task == 'oriented':
                    pred = self.oriented_head(z)
                elif self.current_task == 'classification':
                    pred = self.classification_head(z)
                else:
                    raise ValueError("Task not set")

                predictions_list.append(pred)

            # Stack predictions: (num_mc_samples, batch_size, num_outputs)
            predictions_stacked = torch.stack(predictions_list, dim=0)

            # Compute mean and variance across MC samples
            mean_predictions = predictions_stacked.mean(dim=0)
            uncertainty = predictions_stacked.var(dim=0)

            return {
                'predictions': mean_predictions,
                'uncertainty': uncertainty,
                'num_mc_samples': num_mc_samples
            }
        else:
            # Single forward pass (no uncertainty computation)
            x = self.bayesian_head(features, sample=sample)

            # Task-specific output
            if self.current_task == 'detection':
                predictions = self.detection_head(x)
            elif self.current_task == 'segmentation':
                predictions = self.segmentation_head(x)
            elif self.current_task == 'pose':
                predictions = self.pose_head(x)
            elif self.current_task == 'oriented':
                predictions = self.oriented_head(x)
            elif self.current_task == 'classification':
                predictions = self.classification_head(x)
            else:
                raise ValueError("Task not set")

            return {
                'predictions': predictions,
                'uncertainty': torch.zeros_like(predictions),
                'num_mc_samples': 1
            }

    def predict_with_uncertainty(self, x: torch.Tensor, T: int = 10):
        """
        Monte Carlo inference with detailed uncertainty estimates.
        """
        self.eval()

        preds = []
        with torch.no_grad():
            for _ in range(T):
                output = self.forward(x, sample=True, num_mc_samples=1, return_uncertainty=False)
                preds.append(output['predictions'])

        preds = torch.stack(preds)  # (T, B, D)

        if self.current_task == 'classification':
            probs = torch.softmax(preds, dim=-1)
            return {
                'mean': probs.mean(0),
                'uncertainty': probs.var(0),
                'confidence': probs.mean(0).max(dim=1)[0]
            }

        elif self.current_task == 'detection':
            bbox = preds[..., :4]
            logits = preds[..., 4:]

            probs = torch.softmax(logits, dim=-1)

            return {
                'bbox_mean': bbox.mean(0),
                'bbox_uncertainty': bbox.var(0),
                'class_probs_mean': probs.mean(0),
                'class_uncertainty': probs.var(0),
                'confidence': probs.mean(0).max(dim=1)[0]
            }

        elif self.current_task == 'oriented':
            bbox = preds[..., :4]
            angle = preds[..., 4:5]
            logits = preds[..., 5:]

            probs = torch.softmax(logits, dim=-1)

            return {
                'bbox_mean': bbox.mean(0),
                'bbox_uncertainty': bbox.var(0),
                'angle_mean': angle.mean(0),
                'angle_uncertainty': angle.var(0),
                'class_probs_mean': probs.mean(0),
                'class_uncertainty': probs.var(0),
                'confidence': probs.mean(0).max(dim=1)[0]
            }

        elif self.current_task == 'segmentation':
            probs = torch.sigmoid(preds)
            return {
                'mean': probs.mean(0),
                'uncertainty': probs.var(0)
            }

        elif self.current_task == 'pose':
            return {
                'mean': preds.mean(0),
                'uncertainty': preds.var(0)
            }

        else:
            raise ValueError("Unknown task")

    def get_kl_divergence(self):
        return self.bayesian_head.get_kl_divergence()


**Task-Specific Loss Functions**

In [8]:
import torch
import torch.nn as nn
import torch.nn.functional as F


# =========================================================
# TASK-SPECIFIC LOSS
# =========================================================
class TaskSpecificLoss(nn.Module):
    def __init__(self, task: str, num_classes: int = 80):
        super().__init__()
        self.task = task
        self.num_classes = num_classes

    def forward(
        self,
        predictions: torch.Tensor,
        targets: torch.Tensor,
        kl_divergence: torch.Tensor = None,
        kl_weight: float = 0.01
    ) -> torch.Tensor:

        if self.task == 'detection':
            task_loss = self._detection_loss(predictions, targets)

        elif self.task == 'segmentation':
            task_loss = self._segmentation_loss(predictions, targets)

        elif self.task == 'pose':
            task_loss = self._pose_loss(predictions, targets)

        elif self.task == 'oriented':
            task_loss = self._oriented_loss(predictions, targets)

        elif self.task == 'classification':
            task_loss = self._classification_loss(predictions, targets)

        else:
            raise ValueError(f"Unknown task: {self.task}")

        if kl_divergence is not None:
            task_loss = task_loss + kl_weight * kl_divergence

        return task_loss

    # =====================================================
    # DETECTION
    # =====================================================
    def _detection_loss(self, predictions, targets):
        bbox_pred = predictions[:, :4]
        bbox_target = targets[:, :4]

        bbox_loss = F.smooth_l1_loss(bbox_pred, bbox_target)

        class_pred = predictions[:, 4:]

        if targets.dim() == 2:
            class_target = targets[:, 4:]
            if class_target.shape[1] == self.num_classes:
                class_target = class_target.argmax(dim=1)
            else:
                raise ValueError("Invalid class target shape")
        else:
            raise ValueError("Targets must be 2D")

        class_loss = F.cross_entropy(class_pred, class_target)

        return bbox_loss * 0.5 + class_loss

    # =====================================================
    # SEGMENTATION
    # =====================================================
    def _segmentation_loss(self, predictions, targets):
        bce = F.binary_cross_entropy_with_logits(predictions, targets)

        probs = torch.sigmoid(predictions)

        intersection = (probs * targets).sum(dim=1)
        union = probs.sum(dim=1) + targets.sum(dim=1)

        dice = 1 - ((2 * intersection + 1e-8) / (union + 1e-8))
        dice = dice.mean()

        return 0.5 * bce + 0.5 * dice

    # =====================================================
    # POSE
    # =====================================================
    def _pose_loss(self, predictions, targets):
        mse = F.mse_loss(predictions, targets)
        smooth = F.smooth_l1_loss(predictions, targets)
        return 0.7 * mse + 0.3 * smooth

    # =====================================================
    # ORIENTED DETECTION
    # =====================================================
    def _oriented_loss(self, predictions, targets):
        bbox_loss = F.smooth_l1_loss(predictions[:, :4], targets[:, :4])

        angle_pred = predictions[:, 4]
        angle_target = targets[:, 4]

        angle_diff = torch.atan2(
            torch.sin(angle_pred - angle_target),
            torch.cos(angle_pred - angle_target)
        )

        angle_loss = torch.mean(angle_diff ** 2)

        class_pred = predictions[:, 5:]

        class_target = targets[:, 5:]
        if class_target.shape[1] == self.num_classes:
            class_target = class_target.argmax(dim=1)

        class_loss = F.cross_entropy(class_pred, class_target)

        return 0.4 * bbox_loss + 0.3 * angle_loss + 0.3 * class_loss

    # =====================================================
    # CLASSIFICATION
    # =====================================================
    def _classification_loss(self, predictions, targets):
        if targets.dim() == 2:
            targets = targets.argmax(dim=1)

        return F.cross_entropy(predictions, targets)


# =========================================================
# EWC REGULARIZER
# =========================================================
class EWCRegularizer(nn.Module):
    def __init__(self, model: nn.Module, lambda_ewc: float = 0.4):
        super().__init__()
        self.model = model
        self.lambda_ewc = lambda_ewc

        self.fisher_information = {}
        self.previous_weights = {}

    def compute_fisher_information(
        self,
        dataloader,
        loss_fn,
        device
    ):
        self.model.eval()

        fisher = {
            name: torch.zeros_like(param, device=device)
            for name, param in self.model.named_parameters()
            if param.requires_grad
        }

        total_samples = 0

        for images, targets in dataloader:
            images = images.to(device)
            targets = targets.to(device)

            self.model.zero_grad()

            outputs = self.model(images, sample=False)
            loss = loss_fn(outputs, targets)

            loss.backward()

            for name, param in self.model.named_parameters():
                if param.grad is not None:
                    fisher[name] += (param.grad.detach() ** 2)

            total_samples += 1

        for name in fisher:
            fisher[name] /= max(total_samples, 1)
            fisher[name] += 1e-8  # stability

        self.fisher_information = fisher
        self._save_previous_weights()

    def _save_previous_weights(self):
        self.previous_weights = {
            name: param.detach().clone()
            for name, param in self.model.named_parameters()
            if param.requires_grad
        }

    def get_ewc_loss(self):
        if not self.fisher_information:
            return torch.tensor(0.0, device=next(self.model.parameters()).device)

        loss = torch.zeros(1, device=next(self.model.parameters()).device)

        for name, param in self.model.named_parameters():
            if name in self.fisher_information:
                fisher = self.fisher_information[name]
                prev = self.previous_weights[name]

                loss += torch.sum(fisher * (param - prev) ** 2)

        return (self.lambda_ewc / 2.0) * loss

**Training Loop with Best Model Saving**

In [9]:
import torch
import torch.optim as optim
from torch.utils.data import DataLoader
import os
from pathlib import Path
from tqdm import tqdm
from collections import defaultdict


class B_YOLOTrainer:
    def __init__(
        self,
        model,
        tasks,
        device=None,
        checkpoint_dir='./checkpoints'
    ):
        self.model = model
        self.tasks = tasks
        self.device = device or torch.device('cuda' if torch.cuda.is_available() else 'cpu')

        self.checkpoint_dir = Path(checkpoint_dir)
        self.checkpoint_dir.mkdir(parents=True, exist_ok=True)

        self.ewc_regularizer = EWCRegularizer(model)

        self.best_metrics = defaultdict(lambda: {'score': 0.0, 'model_path': None})
        self.training_history = defaultdict(list)

    # =====================================================
    # MAIN TRAINING LOOP
    # =====================================================
    def train_sequential_tasks(
        self,
        task_dataloaders,
        num_epochs_per_task=50,
        learning_rate=1e-3,
        kl_weight=0.01,
        lambda_ewc=0.4,
        mc_samples_train=1,
        mc_samples_val=3
    ):
        self.model.to(self.device)

        for task_idx, task in enumerate(self.tasks):

            print(f"\n{'='*60}")
            print(f"Task {task_idx+1}/{len(self.tasks)}: {task.upper()}")
            print(f"{'='*60}")

            self.model.set_task(task)

            loss_fn = TaskSpecificLoss(task, self.model.num_classes)

            optimizer = optim.Adam(
                filter(lambda p: p.requires_grad, self.model.parameters()),
                lr=learning_rate
            )

            scheduler = optim.lr_scheduler.ReduceLROnPlateau(
                optimizer, mode='max', factor=0.5, patience=5
            )

            train_loader = task_dataloaders[task]['train']
            val_loader = task_dataloaders[task]['val']

            best_val_metric = -float('inf')
            patience_counter = 0
            patience = 10

            for epoch in range(num_epochs_per_task):

                train_loss = self._train_epoch(
                    train_loader,
                    loss_fn,
                    optimizer,
                    kl_weight,
                    lambda_ewc if task_idx > 0 else 0.0,
                    mc_samples=mc_samples_train
                )

                val_metric = self._validate(
                    val_loader,
                    task,
                    mc_samples=mc_samples_val
                )

                self.training_history[task].append({
                    'epoch': epoch,
                    'train_loss': train_loss,
                    'val_metric': val_metric
                })

                print(f"Epoch {epoch+1}/{num_epochs_per_task} | "
                      f"Loss: {train_loss:.4f} | Val: {val_metric:.4f}")

                if val_metric > best_val_metric:
                    best_val_metric = val_metric
                    patience_counter = 0
                    self._save_checkpoint(task, epoch, val_metric)
                else:
                    patience_counter += 1

                scheduler.step(val_metric)

                if patience_counter >= patience:
                    print("Early stopping triggered")
                    break


            if task_idx < len(self.tasks) - 1:
                print("\nComputing Fisher Information...")
                self.ewc_regularizer.compute_fisher_information(
                    train_loader,
                    loss_fn,
                    self.device
                )

    # =====================================================
    # TRAIN EPOCH
    # =====================================================
    def _train_epoch(
        self,
        dataloader,
        loss_fn,
        optimizer,
        kl_weight,
        ewc_weight,
        mc_samples=1
    ):
        self.model.train()

        total_loss = 0.0

        num_batches = len(dataloader)

        for images, targets in tqdm(dataloader, desc="Training", leave=False):

            images = images.to(self.device)
            targets = targets.to(self.device)

            optimizer.zero_grad()

            # Get predictions and uncertainty
            output = self.model(
                images,
                sample=True,
                num_mc_samples=mc_samples,
                return_uncertainty=True
            )
            predictions = output['predictions']

            # KL divergence
            kl_div = self.model.get_kl_divergence() / num_batches

            loss = loss_fn(predictions, targets, kl_div, kl_weight)

            # EWC regularization
            if ewc_weight > 0:
                loss = loss + ewc_weight * self.ewc_regularizer.get_ewc_loss()

            loss.backward()

            torch.nn.utils.clip_grad_norm_(self.model.parameters(), 1.0)

            optimizer.step()

            total_loss += loss.item()

        return total_loss / num_batches

    # =====================================================
    # VALIDATION
    # =====================================================
    def _validate(self, dataloader, task, mc_samples=1):
        self.model.eval()

        total_metric = 0.0
        num_batches = 0

        with torch.no_grad():
            for images, targets in dataloader:

                images = images.to(self.device)
                targets = targets.to(self.device)

                # Get predictions and uncertainty
                output = self.model(
                    images,
                    sample=True,
                    num_mc_samples=mc_samples,
                    return_uncertainty=True
                )
                predictions = output['predictions']
                uncertainty = output['uncertainty']

                metric = self._compute_task_metric(predictions, targets, task)

                total_metric += metric
                num_batches += 1

        return total_metric / max(num_batches, 1)

    # =====================================================
    # METRICS
    # =====================================================
    def _compute_task_metric(self, predictions, targets, task):

        if task in ['detection', 'oriented']:
            iou = self._compute_iou(predictions[:, :4], targets[:, :4])
            return iou.mean().item()

        elif task == 'segmentation':
            probs = torch.sigmoid(predictions)

            intersection = (probs * targets).sum(dim=1)
            union = probs.sum(dim=1) + targets.sum(dim=1)

            dice = (2 * intersection + 1e-8) / (union + 1e-8)
            return dice.mean().item()

        elif task == 'pose':
            dist = torch.norm(predictions - targets, dim=1).mean()
            return -dist.item()

        elif task == 'classification':
            preds = predictions.argmax(dim=1)
            targets = targets.argmax(dim=1) if targets.dim() == 2 else targets
            return (preds == targets).float().mean().item()

        return 0.0

    # =====================================================
    # IOU
    # =====================================================
    def _compute_iou(self, pred, target):

        x1 = torch.max(pred[:, 0], target[:, 0])
        y1 = torch.max(pred[:, 1], target[:, 1])
        x2 = torch.min(pred[:, 2], target[:, 2])
        y2 = torch.min(pred[:, 3], target[:, 3])

        inter = (x2 - x1).clamp(min=0) * (y2 - y1).clamp(min=0)

        area_p = (pred[:, 2] - pred[:, 0]) * (pred[:, 3] - pred[:, 1])
        area_t = (target[:, 2] - target[:, 0]) * (target[:, 3] - target[:, 1])

        union = area_p + area_t - inter

        return inter / (union + 1e-8)

    # =====================================================
    # CHECKPOINTING
    # =====================================================
    def _save_checkpoint(self, task, epoch, metric):

        path = self.checkpoint_dir / f"b_yolo_{task}_best.pt"

        torch.save({
            'epoch': epoch,
            'model_state_dict': self.model.state_dict(),
            'metric': metric,
            'fisher': self.ewc_regularizer.fisher_information,
            'prev_weights': self.ewc_regularizer.previous_weights
        }, path)

        self.best_metrics[task]['score'] = metric
        self.best_metrics[task]['model_path'] = str(path)

        print(f" Saved best {task} model: {metric:.4f}")


In [10]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset


class DummyBackbone(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Conv2d(3, 512, 1)
    def forward(self, x):
        return self.conv(x)

# 2. Initialize Model
backbone = DummyBackbone()
model = B_YOLO(backbone, num_classes=10)


def create_dummy_loader(num_samples=10, input_shape=(3, 224, 224), num_classes=10):
    x = torch.randn(num_samples, *input_shape)
    y = torch.randint(0, num_classes, (num_samples,))
    dataset = TensorDataset(x, y)
    return DataLoader(dataset, batch_size=2)

task_dataloaders = {
    'classification': {
        'train': create_dummy_loader(),
        'val': create_dummy_loader()
    }
}


trainer = B_YOLOTrainer(model, tasks=['classification'])


try:
    print("Starting smoke test...")
    trainer.train_sequential_tasks(
        task_dataloaders,
        num_epochs_per_task=1,
        learning_rate=1e-4,
        mc_samples_train=1,  # Single sample during training
        mc_samples_val=2     # Multiple samples during validation for uncertainty
    )
    print("\n Success! The pipeline runs correctly.")

    # Test uncertainty output
    print("\nTesting uncertainty output...")
    model.eval()
    sample_input = torch.randn(2, 3, 224, 224)

    # Test with single sample (no uncertainty)
    output_single = model(sample_input, num_mc_samples=1, return_uncertainty=True)
    print(f" Single forward pass:")
    print(f"  - Predictions shape: {output_single['predictions'].shape}")
    print(f"  - Uncertainty shape: {output_single['uncertainty'].shape}")
    print(f"  - Uncertainty (should be mostly zeros): {output_single['uncertainty'].mean().item():.6f}")

    # Test with multiple samples (with uncertainty)
    output_multi = model(sample_input, num_mc_samples=5, return_uncertainty=True)
    print(f"\n Multiple MC forward passes (5 samples):")
    print(f"  - Predictions shape: {output_multi['predictions'].shape}")
    print(f"  - Uncertainty shape: {output_multi['uncertainty'].shape}")
    print(f"  - Uncertainty (should be non-zero): {output_multi['uncertainty'].mean().item():.6f}")

except Exception as e:
    print(f"\nAn error occurred: {e}")
    import traceback
    traceback.print_exc()


Starting smoke test...

Task 1/1: CLASSIFICATION


Epoch 1/1 | Loss: 662.6250 | Val: 0.1000
 Saved best classification model: 0.1000

 Success! The pipeline runs correctly.

Testing uncertainty output...
 Single forward pass:
  - Predictions shape: torch.Size([2, 10])
  - Uncertainty shape: torch.Size([2, 10])
  - Uncertainty (should be mostly zeros): 0.000000

 Multiple MC forward passes (5 samples):
  - Predictions shape: torch.Size([2, 10])
  - Uncertainty shape: torch.Size([2, 10])
  - Uncertainty (should be non-zero): 0.087516


In [11]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import numpy as np

print("="*70)
print("COMPREHENSIVE DUMMY INPUT & OUTPUT DEMONSTRATION")
print("="*70)

# Test 1: CLASSIFICATION TASK
print("\n" + "="*70)
print("TEST 1: CLASSIFICATION TASK")
print("="*70)

model.eval()
model.set_task('classification')

# Create dummy input: batch of 3 images (3, 224, 224)
dummy_input = torch.randn(3, 3, 224, 224)
print(f"\nInput shape: {dummy_input.shape} (batch_size=3, channels=3, height=224, width=224)")

# Single forward pass (no uncertainty)
with torch.no_grad():
    output_single = model(dummy_input, num_mc_samples=1, return_uncertainty=False)

print(f"\n--- Single Forward Pass (num_mc_samples=1) ---")
print(f"Predictions shape: {output_single['predictions'].shape}")
print(f"Predictions (probabilities after softmax):")
probs = torch.softmax(output_single['predictions'], dim=1)
for i in range(3):
    print(f"  Sample {i+1}: {probs[i].detach().numpy()}")

# Multiple forward passes with uncertainty
with torch.no_grad():
    output_multi = model(dummy_input, num_mc_samples=10, return_uncertainty=True)

print(f"\n--- Multiple Forward Passes (num_mc_samples=10) with Uncertainty ---")
print(f"Mean Predictions shape: {output_multi['predictions'].shape}")
print(f"Uncertainty shape: {output_multi['uncertainty'].shape}")
print(f"\nPredictions (mean across MC samples):")
probs_mean = torch.softmax(output_multi['predictions'], dim=1)
for i in range(3):
    print(f"  Sample {i+1}: {probs_mean[i].detach().numpy()}")

print(f"\nUncertainty scores (variance across MC samples):")
for i in range(3):
    print(f"  Sample {i+1}: {output_multi['uncertainty'][i].detach().numpy()}")
    print(f"             Mean uncertainty: {output_multi['uncertainty'][i].mean().item():.6f}")

# Test 2: DETECTION TASK
print("\n" + "="*70)
print("TEST 2: DETECTION TASK")
print("="*70)

model.set_task('detection')

with torch.no_grad():
    output_det = model(dummy_input, num_mc_samples=5, return_uncertainty=True)

print(f"\nInput shape: {dummy_input.shape}")
print(f"Output predictions shape: {output_det['predictions'].shape}")
print(f"Output uncertainty shape: {output_det['uncertainty'].shape}")
print(f"  - First 4 values per sample: bounding boxes (x1, y1, x2, y2)")
print(f"  - Remaining {output_det['predictions'].shape[1]-4} values: class logits")

print(f"\nDetection Output for Sample 1:")
bbox = output_det['predictions'][0, :4]
classes = output_det['predictions'][0, 4:]
print(f"  Bounding Box: {bbox.detach().numpy()}")
print(f"  Class Logits: {classes.detach().numpy()}")

print(f"\nUncertainty for Sample 1:")
unc_bbox = output_det['uncertainty'][0, :4]
unc_classes = output_det['uncertainty'][0, 4:]
print(f"  BBox Uncertainty: {unc_bbox.detach().numpy()}")
print(f"  Class Uncertainty: {unc_classes.detach().numpy()}")
print(f"  Mean Uncertainty: {output_det['uncertainty'][0].mean().item():.6f}")

# Test 3: Monte Carlo Inference (detailed)
print("\n" + "="*70)
print("TEST 3: DETAILED MC INFERENCE")
print("="*70)

model.set_task('classification')

with torch.no_grad():
    # Single sample inference
    sample = dummy_input[:1]  # Take first sample
    mc_output = model.predict_with_uncertainty(sample, T=20)

print(f"\nUsing T=20 MC forward passes on 1 sample:")
print(f"Mean predictions shape: {mc_output['mean'].shape}")
print(f"Uncertainty shape: {mc_output['uncertainty'].shape}")
print(f"Confidence shape: {mc_output['confidence'].shape}")

print(f"\nMean class probabilities: {mc_output['mean'][0].detach().numpy()}")
print(f"Uncertainty per class: {mc_output['uncertainty'][0].detach().numpy()}")
print(f"Confidence (max prob): {mc_output['confidence'][0].item():.4f}")
print(f"Predicted class: {mc_output['mean'][0].argmax().item()}")

print("\n" + "="*70)
print(" All tests completed successfully!")
print("="*70)


COMPREHENSIVE DUMMY INPUT & OUTPUT DEMONSTRATION

TEST 1: CLASSIFICATION TASK

Input shape: torch.Size([3, 3, 224, 224]) (batch_size=3, channels=3, height=224, width=224)

--- Single Forward Pass (num_mc_samples=1) ---
Predictions shape: torch.Size([3, 10])
Predictions (probabilities after softmax):
  Sample 1: [0.07693281 0.17247987 0.10224172 0.0888525  0.12534519 0.03313423
 0.08384464 0.15848291 0.08976473 0.06892136]
  Sample 2: [0.07672662 0.17360277 0.10219328 0.08865663 0.12564239 0.0331422
 0.08493678 0.15759929 0.08849237 0.06900771]
  Sample 3: [0.07704789 0.1722995  0.10264994 0.08847646 0.12568745 0.03302672
 0.08439481 0.1581289  0.08959276 0.06869551]

--- Multiple Forward Passes (num_mc_samples=10) with Uncertainty ---
Mean Predictions shape: torch.Size([3, 10])
Uncertainty shape: torch.Size([3, 10])

Predictions (mean across MC samples):
  Sample 1: [0.11250249 0.16285175 0.06604688 0.18876189 0.06802509 0.05362395
 0.10104284 0.11070519 0.06507935 0.07136057]
  Sample

In [13]:
# ============================================================
# SIMPLE DUMMY INPUT → MODEL OUTPUT DEMONSTRATION
# ============================================================

print("\n" + "="*80)
print("SIMPLE MODEL INPUT/OUTPUT EXAMPLE")
print("="*80)

# Initialize model
backbone = DummyBackbone()
model = B_YOLO(backbone, num_classes=5)
model.eval()

# ============================================================
# EXAMPLE 1: CLASSIFICATION TASK
# ============================================================
print("\n" + "-"*80)
print("EXAMPLE 1: CLASSIFICATION TASK")
print("-"*80)

model.set_task('classification')

# Create 2 dummy images
dummy_batch = torch.randn(2, 3, 224, 224)
print(f"\n INPUT: Created dummy batch")
print(f"  Shape: {dummy_batch.shape}")
print(f"  (batch_size=2, channels=3, height=224, width=224)")

# Get single prediction
with torch.no_grad():
    output = model(dummy_batch, num_mc_samples=1, return_uncertainty=False)

print(f"\n OUTPUT: Model predictions")
print(f"  Raw logits shape: {output['predictions'].shape}")
print(f"  Raw logits for image 1: {output['predictions'][0].numpy()}")
print(f"  Raw logits for image 2: {output['predictions'][1].numpy()}")

# Convert to probabilities
probs = torch.softmax(output['predictions'], dim=1)
print(f"\n PROBABILITIES (softmax):")
for i, prob in enumerate(probs):
    pred_class = prob.argmax().item()
    pred_conf = prob.max().item()
    print(f"  Image {i+1}:")
    print(f"    All class probs: {prob.numpy()}")
    print(f"    Predicted class: {pred_class}, Confidence: {pred_conf:.4f}")

# ============================================================
# EXAMPLE 2: CLASSIFICATION WITH UNCERTAINTY
# ============================================================
print("\n" + "-"*80)
print("EXAMPLE 2: CLASSIFICATION WITH UNCERTAINTY (7 MC samples)")
print("-"*80)

with torch.no_grad():
    output = model(dummy_batch, num_mc_samples=7, return_uncertainty=True)

print(f"\n OUTPUT with Uncertainty:")
print(f"  Predictions shape: {output['predictions'].shape}")
print(f"  Uncertainty shape: {output['uncertainty'].shape}")

probs_mean = torch.softmax(output['predictions'], dim=1)
print(f"\n MEAN PREDICTIONS:")
for i, prob in enumerate(probs_mean):
    print(f"  Image {i+1}: {prob.numpy()}")

print(f"\n UNCERTAINTY SCORES:")
for i, unc in enumerate(output['uncertainty']):
    print(f"  Image {i+1}: {unc.numpy()}")
    print(f"           Average uncertainty: {unc.mean().item():.6f}")

# ============================================================
# EXAMPLE 3: DETECTION TASK
# ============================================================
print("\n" + "-"*80)
print("EXAMPLE 3: DETECTION TASK (with 3 MC samples)")
print("-"*80)

model.set_task('detection')

with torch.no_grad():
    output = model(dummy_batch, num_mc_samples=3, return_uncertainty=True)

print(f"\n OUTPUT: Detection predictions")
print(f"  Predictions shape: {output['predictions'].shape}")
print(f"  (2 images, 4 bbox values + 5 class logits = 9 total)")

pred = output['predictions']
print(f"\n IMAGE 1 DETECTION OUTPUT:")
print(f"  Bounding Box [x1, y1, x2, y2]: {pred[0, :4].numpy()}")
print(f"  Class Logits: {pred[0, 4:].numpy()}")

# Convert to probabilities
class_probs = torch.softmax(pred[:, 4:], dim=1)
print(f"\n IMAGE 1 CLASS PROBABILITIES:")
print(f"  Probabilities: {class_probs[0].numpy()}")
print(f"  Predicted class: {class_probs[0].argmax().item()}")

print(f"\n IMAGE 1 UNCERTAINTY:")
unc = output['uncertainty']
print(f"  BBox uncertainty [x1, y1, x2, y2]: {unc[0, :4].numpy()}")
print(f"  Class uncertainty: {unc[0, 4:].numpy()}")
print(f"  Mean uncertainty: {unc[0].mean().item():.6f}")

# ============================================================
# EXAMPLE 4: POSE ESTIMATION TASK
# ============================================================
print("\n" + "-"*80)
print("EXAMPLE 4: POSE ESTIMATION TASK")
print("-"*80)

model.set_task('pose')

with torch.no_grad():
    output = model(dummy_batch, num_mc_samples=1, return_uncertainty=False)

print(f"\n OUTPUT: Pose predictions")
print(f"  Output shape: {output['predictions'].shape}")
print(f"  (2 images, 17 keypoints × 2 coordinates = 34 values)")

pose_pred = output['predictions']
print(f"\n IMAGE 1 POSE KEYPOINTS:")
print(f"  All keypoints (x, y coords): {pose_pred[0].numpy()}")
print(f"  First keypoint (x, y): [{pose_pred[0, 0].item():.4f}, {pose_pred[0, 1].item():.4f}]")
print(f"  Second keypoint (x, y): [{pose_pred[0, 2].item():.4f}, {pose_pred[0, 3].item():.4f}]")

print("\n" + "="*80)
print(" ALL EXAMPLES COMPLETED -  NOW HAVE MODEL OUTPUTS that too with uncertainity scores @ Group 2!")
print("="*80)


SIMPLE MODEL INPUT/OUTPUT EXAMPLE

--------------------------------------------------------------------------------
EXAMPLE 1: CLASSIFICATION TASK
--------------------------------------------------------------------------------

 INPUT: Created dummy batch
  Shape: torch.Size([2, 3, 224, 224])
  (batch_size=2, channels=3, height=224, width=224)

 OUTPUT: Model predictions
  Raw logits shape: torch.Size([2, 5])
  Raw logits for image 1: [ 0.44945306 -0.43454748  0.40634513  0.6081164  -0.66867197]
  Raw logits for image 2: [ 0.45677358 -0.42910975  0.40907696  0.61440253 -0.665115  ]

 PROBABILITIES (softmax):
  Image 1:
    All class probs: [0.25841329 0.10675747 0.2475103  0.30284572 0.08447327]
    Predicted class: 3, Confidence: 0.3028
  Image 2:
    All class probs: [0.25892186 0.10676637 0.24686208 0.30312806 0.08432158]
    Predicted class: 3, Confidence: 0.3031

--------------------------------------------------------------------------------
EXAMPLE 2: CLASSIFICATION WITH UNCER

Model Output Summary:

TEST 1: CLASSIFICATION (single forward pass)
Input: 2 dummy images (2×3×224×224)
Output: 5 class predictions (logits)
Image 1: [-0.392, 0.081, 0.062, -0.743, -0.379]
Image 2: [-0.392, 0.080, 0.062, -0.744, -0.384]
Probabilities:
Image 1: [0.170, 0.272, 0.267, 0.119, 0.172] (class 1 has highest confidence ~0.27)

TEST 2: CLASSIFICATION WITH UNCERTAINTY (5 MC samples)
Mean predictions: [-0.412, 0.785, -0.145, 0.374, 0.270]
Uncertainty scores: [0.035, 0.110, 0.092, 0.185, 0.069]
Highest uncertainty in class 3 (0.185)
Lowest uncertainty in class 0 (0.035)

TEST 3: DETECTION OUTPUT
Bounding Box: [-0.581, 0.535, 0.261, -0.486] (x1, y1, x2, y2)
Class Logits: [0.979, 0.403, 0.748, -0.692, -0.208]
BBox Uncertainty: [0.022, 0.271, 0.173, 0.005]
Class Uncertainty: [0.075, 0.065, 0.102, 0.325, 0.110]